# Assignment 2: Word Representations

**Project:** Job Skill Analysis

**Dataset:** Cleaned Job Skill Set Dataset

## Part A: Count-Based Word Representations

This notebook implements Bag of Words, TF-IDF, and N-gram representations using the cleaned corpus from Assignment 1.


In [11]:
import pandas as pd
from pathlib import Path

# Path to the cleaned dataset
dataset_path = Path("../data/processed/cleaned_job_skill_dataset.parquet")

# Load the Parquet dataset
df = pd.read_parquet(dataset_path)

# Display the dataset's dimensions
print("Dataset shape:", df.shape)

# Display column names
print("\nColumns:")
print(df.columns.tolist())

# Show the first five records
display(df.head())

Dataset shape: (1167, 7)

Columns:
['job_id', 'category', 'job_title', 'job_description', 'job_skill_set', 'word_count', 'cleaned_description']


,job_id,category,job_title,job_description,job_skill_set,word_count,cleaned_description
0,3902668440,HR,Sr Human Resource Generalist,SUMMARY\nTHE SR. HR GENERALIST PROVIDES HR EXP...,"['employee relations', 'talent acquisition', '...",583,summary the sr hr generalist provide hr expert...
1,3905823748,HR,Human Resources Manager,BE PART OF A STELLAR TEAM AT YSB AS THE MANAGE...,"['Talent Acquisition', 'Employee Performance M...",323,be part of a stellar team at ysb as the manage...
2,3905854799,HR,Director of Human Resources,OUR CLIENT IS A THRIVING ORGANIZATION OFFERING...,"['Human Resources Management', 'Recruitment', ...",666,our client be a thrive organization offer a un...
3,3905834061,HR,Chief Human Resources Officer,JOB TITLE: CHIEF HUMAN RESOURCES OFFICER (CHRO...,"['talent management', 'organizational developm...",416,job title chief human resource officer chro)in...
4,3906250451,HR,Human Resources Generalist (Hybrid Role),DESCRIPTION\n\n WHO WE ARE \n\nAVI-SPL IS A DI...,"['Microsoft Office', 'Data analysis', 'Employe...",604,description who we be avi spl be a digital ena...


In [64]:
print("Dataset shape:", df.shape)
print("Number of documents:", len(corpus))
print("Total tokens:", sum(len(doc.split()) for doc in corpus))
print("Unique words:", len(set(" ".join(corpus).split())))

display(df[["cleaned_description"]].head(3))

Dataset shape: (1167, 7)
Number of documents: 1167
Total tokens: 613180
Unique words: 26485


,cleaned_description
0,summary the sr hr generalist provide hr expert...
1,be part of a stellar team at ysb as the manage...
2,our client be a thrive organization offer a un...


In [12]:
# Select the cleaned text column from Assignment 1
corpus = df["cleaned_description"].fillna("").astype(str)

print("Number of documents:", len(corpus))

print("\nFirst cleaned document:")
print(corpus.iloc[0][:500])

print("\nEmpty documents:", (corpus.str.strip() == "").sum())

Number of documents: 1167

First cleaned document:
summary the sr hr generalist provide hr expertise across various function include employee relation talent acquisition performance management payroll compensation and benefit the sr hr generalist have responsibility that be strategic administrative and consultative this role provide consultative hr business partner support to assign business unit across all aspect of the employee life cycle this role also support hr operation with weekly payroll processing the sr hr generalist be also responsibl

Empty documents: 0


In [13]:
from sklearn.feature_extraction.text import CountVectorizer

# Create the Bag of Words vectorizer
bow_vectorizer = CountVectorizer()

# Learn vocabulary and transform documents into word counts
bow_matrix = bow_vectorizer.fit_transform(corpus)

print("Bag of Words matrix shape:", bow_matrix.shape)
print("Vocabulary size:", len(bow_vectorizer.vocabulary_))

Bag of Words matrix shape: (1167, 18635)
Vocabulary size: 18635


In [14]:
# Total number of cells in the document-term matrix
total_cells = bow_matrix.shape[0] * bow_matrix.shape[1]

# Number of nonzero cells
nonzero_cells = bow_matrix.nnz

# Calculate percentage of zero cells
sparsity = (1 - nonzero_cells / total_cells) * 100

print("Total matrix cells:", total_cells)
print("Nonzero cells:", nonzero_cells)
print(f"Sparsity: {sparsity:.2f}%")

Total matrix cells: 21747045
Nonzero cells: 305409
Sparsity: 98.60%


In [15]:
# Get vocabulary terms
feature_names = bow_vectorizer.get_feature_names_out()

# Display first 5 documents and first 15 vocabulary terms
bow_preview = pd.DataFrame(
    bow_matrix[:5, :15].toarray(),
    columns=feature_names[:15]
)

display(bow_preview)

,00,000,00008,0001099,00072,000annual,000benefits,000first,000for,000full,000location,000on,004119,004405,004956
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [16]:
import numpy as np

# Sum each word's frequency across all documents
word_counts = np.asarray(bow_matrix.sum(axis=0)).ravel()

# Sort words by frequency
top_indices = np.argsort(word_counts)[::-1][:15]

# Display the 15 most frequent words
top_words = pd.DataFrame({
    "Word": feature_names[top_indices],
    "Frequency": word_counts[top_indices]
})

display(top_words)

,Word,Frequency
0,and,39042
1,to,19649
2,the,17308
3,of,13636
4,be,10726
5,in,10357
6,with,9034
7,for,7184
8,or,5245
9,our,4236


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Learn vocabulary and calculate TF-IDF weights
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF matrix shape: (1167, 18635)
Vocabulary size: 18635


In [18]:
# Calculate total number of matrix cells
total_cells_tfidf = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]

# Count nonzero values
nonzero_tfidf = tfidf_matrix.nnz

# Calculate sparsity
tfidf_sparsity = (1 - nonzero_tfidf / total_cells_tfidf) * 100

print("Total cells:", total_cells_tfidf)
print("Nonzero cells:", nonzero_tfidf)
print(f"TF-IDF sparsity: {tfidf_sparsity:.2f}%")

print(f"BoW sparsity: {sparsity:.2f}%")

Total cells: 21747045
Nonzero cells: 305409
TF-IDF sparsity: 98.60%
BoW sparsity: 98.60%


In [19]:
import numpy as np
import pandas as pd

# Get vocabulary words
tfidf_features = tfidf_vectorizer.get_feature_names_out()

# Select the first job description
doc_index = 0

# Extract TF-IDF weights for this document
doc_vector = tfidf_matrix[doc_index].toarray().ravel()

# Find the 10 highest-weighted words
top_indices = np.argsort(doc_vector)[::-1][:10]

# Create a table
top_tfidf_words = pd.DataFrame({
    "Word": tfidf_features[top_indices],
    "TF-IDF Weight": doc_vector[top_indices]
})

display(top_tfidf_words)

,Word,TF-IDF Weight
0,and,0.473093
1,hr,0.437638
2,payroll,0.228792
3,to,0.175520
4,with,0.168322
5,in,0.150346
6,generalist,0.132724
7,employee,0.126467
8,partner,0.126153
9,of,0.123602


In [20]:
# Get IDF values for all vocabulary terms
idf_values = tfidf_vectorizer.idf_

# Create a table of terms and IDF values
idf_table = pd.DataFrame({
    "Word": tfidf_features,
    "IDF": idf_values
})

# Show 15 words with the lowest IDF
common_words = idf_table.sort_values("IDF").head(15)

display(common_words)

rare_words = idf_table.sort_values("IDF", ascending=False).head(15)

display(rare_words)

,Word,IDF
1499,and,1.003431
17176,to,1.005150
11271,of,1.011193
8261,in,1.012926
16993,the,1.013793
18371,with,1.014662
2247,be,1.026900
6945,for,1.038399
6359,experience,1.088569
11560,or,1.088569


,Word,IDF
18634,ōwner,7.369901
18633,ōninite,7.369901
18632,ōnin,7.369901
2,00008,7.369901
3,0001099,7.369901
4,00072,7.369901
5,000annual,7.369901
18631,état,7.369901
18615,zendesk,7.369901
18614,zealand,7.369901


In [21]:
# Select document 0
doc_index = 0

# Get terms present in the document
present_indices = bow_matrix[doc_index].nonzero()[1]

# Get BoW counts
bow_counts = bow_matrix[doc_index, present_indices].toarray().ravel()

# Get TF-IDF weights
tfidf_weights = tfidf_matrix[doc_index, present_indices].toarray().ravel()

# Create comparison table
comparison_df = pd.DataFrame({
    "Word": bow_vectorizer.get_feature_names_out()[present_indices],
    "BoW Count": bow_counts,
    "TF-IDF Weight": tfidf_weights
})

# Sort by TF-IDF weight
comparison_df = comparison_df.sort_values(
    "TF-IDF Weight",
    ascending=False
)

display(comparison_df.head(15))

,Word,BoW Count,TF-IDF Weight
19,and,54,0.473093
3,hr,20,0.437638
17,payroll,8,0.228792
33,to,20,0.175520
43,with,19,0.168322
76,in,17,0.150346
4,generalist,4,0.132724
11,employee,9,0.126467
31,partner,7,0.126153
38,of,14,0.123602


### TF-IDF Analysis

TF-IDF represents documents by assigning numerical weights to words based on their frequency within a document and their rarity across the corpus.

Unlike Bag of Words, which uses raw word counts, TF-IDF reduces the relative importance of terms that appear across many documents.

The highest-weighted terms in a job description can help identify its distinctive skills, technologies, and responsibilities.

BoW and TF-IDF have the same matrix shape and sparsity when they use the same vocabulary and tokenization rules, but their numerical values are different.

## Part A3: N-gram Representations

N-grams capture sequences of consecutive words.

We compare unigram, unigram + bigram, and unigram + bigram + trigram representations using CountVectorizer and TfidfVectorizer.

The objective is to examine vocabulary growth and the trade-off between richer contextual information and matrix sparsity.

In [22]:
from sklearn.feature_extraction.text import CountVectorizer

# Unigrams only
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
unigram_matrix = unigram_vectorizer.fit_transform(corpus)

# Unigrams + bigrams
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
bigram_matrix = bigram_vectorizer.fit_transform(corpus)

# Unigrams + bigrams + trigrams
trigram_vectorizer = CountVectorizer(ngram_range=(1, 3))
trigram_matrix = trigram_vectorizer.fit_transform(corpus)

print("Unigram shape:", unigram_matrix.shape)
print("Unigram + Bigram shape:", bigram_matrix.shape)
print("Unigram + Bigram + Trigram shape:", trigram_matrix.shape)

Unigram shape: (1167, 18635)
Unigram + Bigram shape: (1167, 205216)
Unigram + Bigram + Trigram shape: (1167, 597126)


In [23]:
def calculate_sparsity(matrix):
    total_cells = matrix.shape[0] * matrix.shape[1]

    if total_cells == 0:
        return 0.0

    zero_percentage = (1 - matrix.nnz / total_cells) * 100

    return zero_percentage


ngram_results = pd.DataFrame({
    "Representation": [
        "Unigrams (1,1)",
        "Unigrams + Bigrams (1,2)",
        "Unigrams + Bigrams + Trigrams (1,3)"
    ],

    "Vocabulary Size": [
        unigram_matrix.shape[1],
        bigram_matrix.shape[1],
        trigram_matrix.shape[1]
    ],

    "Matrix Shape": [
        str(unigram_matrix.shape),
        str(bigram_matrix.shape),
        str(trigram_matrix.shape)
    ],

    "Sparsity (%)": [
        calculate_sparsity(unigram_matrix),
        calculate_sparsity(bigram_matrix),
        calculate_sparsity(trigram_matrix)
    ]
})

display(ngram_results)

,Representation,Vocabulary Size,Matrix Shape,Sparsity (%)
0,"Unigrams (1,1)",18635,"(1167, 18635)",98.595630
1,"Unigrams + Bigrams (1,2)",205216,"(1167, 205216)",99.642899
2,"Unigrams + Bigrams + Trigrams (1,3)",597126,"(1167, 597126)",99.792982


In [24]:
unigram_vocab_size = unigram_matrix.shape[1]
bigram_vocab_size = bigram_matrix.shape[1]
trigram_vocab_size = trigram_matrix.shape[1]

bigram_growth = (
    (bigram_vocab_size - unigram_vocab_size)
    / unigram_vocab_size
) * 100

trigram_growth = (
    (trigram_vocab_size - unigram_vocab_size)
    / unigram_vocab_size
) * 100

print(f"Vocabulary growth with bigrams: {bigram_growth:.2f}%")
print(f"Vocabulary growth with trigrams: {trigram_growth:.2f}%")

Vocabulary growth with bigrams: 1001.24%
Vocabulary growth with trigrams: 3104.33%


In [25]:
# Get all features from the trigram vectorizer
all_features = trigram_vectorizer.get_feature_names_out()

# Extract phrases containing exactly 2 words
bigrams = [
    feature for feature in all_features
    if len(feature.split()) == 2
]

# Extract phrases containing exactly 3 words
trigrams = [
    feature for feature in all_features
    if len(feature.split()) == 3
]

print("First 20 bigrams:")
print(bigrams[:20])

print("\nFirst 20 trigrams:")
print(trigrams[:20])

First 20 bigrams:
['00 00', '00 100', '00 108', '00 111', '00 114', '00 115', '00 130', '00 144', '00 149', '00 156', '00 159', '00 16', '00 161', '00 164', '00 17', '00 18', '00 19', '00 20', '00 209', '00 21']

First 20 trigrams:
['00 00 est', '00 00 the', '00 100 000', '00 108 900', '00 111 000', '00 111 750', '00 114 663', '00 115 000', '00 130 000', '00 144 900', '00 149 500', '00 156 900', '00 159 390', '00 16 00', '00 16 50', '00 161 000', '00 161 400', '00 164 900', '00 17 25', '00 18 75']


In [26]:
import numpy as np

features = trigram_vectorizer.get_feature_names_out()

frequencies = np.asarray(
    trigram_matrix.sum(axis=0)
).ravel()

phrase_df = pd.DataFrame({
    "Phrase": features,
    "Frequency": frequencies
})

# Count words in each feature
phrase_df["N"] = phrase_df["Phrase"].str.split().str.len()

top_bigrams = (
    phrase_df[phrase_df["N"] == 2]
    .sort_values("Frequency", ascending=False)
    .head(10)
)

top_trigrams = (
    phrase_df[phrase_df["N"] == 3]
    .sort_values("Frequency", ascending=False)
    .head(10)
)

print("Top 10 Bigrams:")
display(top_bigrams)

print("Top 10 Trigrams:")
display(top_trigrams)

Top 10 Bigrams:


,Phrase,Frequency,N
349911,of the,1956,2
262333,in the,1649,2
7763,ability to,1623,2
584666,with the,1080,2
546118,to the,957,2
592488,year of,939,2
250863,human resource,916,2
576784,will be,883,2
200684,experience in,823,2
569565,we be,780,2


Top 10 Trigrams:


,Phrase,Frequency,N
86044,be responsible for,432,3
515813,the ability to,424,3
69483,as well as,404,3
592541,year of experience,338,3
81922,be an equal,334,3
190190,equal opportunity employer,326,3
28362,an equal opportunity,317,3
78304,bachelor degree in,297,3
81584,be able to,262,3
8041,ability to work,245,3


In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_ngram_results = []

for ngram_range in [(1, 1), (1, 2), (1, 3)]:

    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range
    )

    matrix = vectorizer.fit_transform(corpus)

    tfidf_ngram_results.append({
        "N-gram Range": str(ngram_range),
        "Vocabulary Size": matrix.shape[1],
        "Matrix Shape": str(matrix.shape),
        "Sparsity (%)": calculate_sparsity(matrix)
    })

tfidf_ngram_df = pd.DataFrame(tfidf_ngram_results)

display(tfidf_ngram_df)

,N-gram Range,Vocabulary Size,Matrix Shape,Sparsity (%)
0,"(1, 1)",18635,"(1167, 18635)",98.595630
1,"(1, 2)",205216,"(1167, 205216)",99.642899
2,"(1, 3)",597126,"(1167, 597126)",99.792982


### N-gram Analysis

The experiment compares three feature configurations:
unigrams (1,1), unigrams and bigrams (1,2), and
unigrams, bigrams and trigrams (1,3).

Increasing the N-gram range allows the model to capture
multiword expressions, which is useful for job descriptions
containing phrases such as "machine learning" and
"project management".

However, longer N-grams increase vocabulary size and
feature dimensionality. They can also increase sparsity
because many phrases appear in relatively few documents.

The final comparison is based on the measured vocabulary
sizes and sparsity percentages.

(1, 1)	vocab size = 18635 , sparsity percentage = 98.595630
(1, 2)	vocab size = 205216, sparsity percentage = 99.642899
(1, 3)	vocab size = 597126, sparsity percentage = 99.792982


## Part A4: Manual TF-IDF and Bigram Calculations

We use a small corpus of four job-related documents to manually
calculate a TF-IDF weight and a bigram frequency.

The manual calculations are verified using scikit-learn.

In [28]:
toy_corpus = [
    "python developer python",
    "python data analyst",
    "java developer",
    "data analyst python"
]

for i, doc in enumerate(toy_corpus, start=1):
    print(f"Document {i}: {doc}")

Document 1: python developer python
Document 2: python data analyst
Document 3: java developer
Document 4: data analyst python


In [29]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer with default L2 normalization
toy_tfidf_vectorizer = TfidfVectorizer()

# Fit and transform toy corpus
toy_tfidf_matrix = toy_tfidf_vectorizer.fit_transform(toy_corpus)

# Find vocabulary index of "python"
python_index = toy_tfidf_vectorizer.vocabulary_["python"]

# Get library result
library_weight = toy_tfidf_matrix[0, python_index]

# Manual calculation
python_idf = np.log((1 + 4) / (1 + 3)) + 1
developer_idf = np.log((1 + 4) / (1 + 2)) + 1

python_raw = 2 * python_idf
developer_raw = 1 * developer_idf

norm = np.sqrt(python_raw**2 + developer_raw**2)

manual_weight = python_raw / norm

print(f"Manual normalized TF-IDF: {manual_weight:.6f}")
print(f"Scikit-learn TF-IDF:       {library_weight:.6f}")
print("Values match:", np.isclose(manual_weight, library_weight))

Manual normalized TF-IDF: 0.850816
Scikit-learn TF-IDF:       0.850816
Values match: True


In [30]:
from sklearn.feature_extraction.text import CountVectorizer

# Extract bigrams only
toy_bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))

toy_bigram_matrix = toy_bigram_vectorizer.fit_transform(toy_corpus)

# Get the index of "data analyst"
bigram_index = toy_bigram_vectorizer.vocabulary_["data analyst"]

# Count occurrences across all documents
bigram_count = toy_bigram_matrix[:, bigram_index].sum()

print("Bigram:", "data analyst")
print("Manual count:", 2)
print("Scikit-learn count:", int(bigram_count))
print("Values match:", int(bigram_count) == 2)

Bigram: data analyst
Manual count: 2
Scikit-learn count: 2
Values match: True


# Part B: Word Embeddings

This section trains Word2Vec CBOW, Word2Vec Skip-gram,
and fastText models using the cleaned job-description corpus.

All models use the same tokenized corpus. CBOW and
Skip-gram use identical hyperparameters to ensure a fair comparison.

In [31]:
import pandas as pd
import numpy as np

from gensim.models import Word2Vec, FastText

In [32]:
tokenized_corpus = [
    document.split()
    for document in corpus
    if document.strip()
]

print("Number of tokenized documents:", len(tokenized_corpus))

if tokenized_corpus:
    print("\nFirst 20 tokens from the first document:")
    print(tokenized_corpus[0][:20])
else:
    print("No documents available for tokenization.")

Number of tokenized documents: 1167

First 20 tokens from the first document:
['summary', 'the', 'sr', 'hr', 'generalist', 'provide', 'hr', 'expertise', 'across', 'various', 'function', 'include', 'employee', 'relation', 'talent', 'acquisition', 'performance', 'management', 'payroll', 'compensation']


In [33]:
total_tokens = sum(len(doc) for doc in tokenized_corpus)

unique_words = set(
    word
    for doc in tokenized_corpus
    for word in doc
)

print("Total tokens:", total_tokens)
print("Unique words:", len(unique_words))
print("Documents used for training:", len(tokenized_corpus))

Total tokens: 613180
Unique words: 26485
Documents used for training: 1167


In [34]:
# Shared hyperparameters for CBOW and Skip-gram
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 2
EPOCHS = 20
SEED = 42

print("Vector size:", VECTOR_SIZE)
print("Window:", WINDOW)
print("Minimum word count:", MIN_COUNT)
print("Epochs:", EPOCHS)

Vector size: 100
Window: 5
Minimum word count: 2
Epochs: 20


In [35]:
cbow_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=0,
    epochs=EPOCHS,
    seed=SEED,
    workers=1
)

print("CBOW training completed!")
print("Vocabulary size:", len(cbow_model.wv))
print("Vector dimensions:", cbow_model.wv.vector_size)

CBOW training completed!
Vocabulary size: 9527
Vector dimensions: 100


In [36]:
query_word = "python"

if query_word in cbow_model.wv.key_to_index:
    vector = cbow_model.wv[query_word]

    print("Word:", query_word)
    print("Vector shape:", vector.shape)
    print("First 10 vector values:")
    print(vector[:10])
else:
    print(f"'{query_word}' is not in the trained vocabulary.")

Word: python
Vector shape: (100,)
First 10 vector values:
[-0.35948035  0.09616979 -0.5334015   0.8446815  -0.01826361  0.21093883
 -0.31196213 -0.02486558  0.17341152 -0.03214784]


In [37]:
if query_word in cbow_model.wv.key_to_index:
    similar_words = cbow_model.wv.most_similar(
        query_word,
        topn=10
    )

    for word, similarity in similar_words:
        print(f"{word:20s} {similarity:.4f}")

sql                  0.8901
scripting            0.8441
java                 0.8322
tableau              0.8281
powerbi              0.8037
google               0.8016
language             0.7866
onedrive             0.7857
spark                0.7637
hubspot              0.7624


In [38]:
skipgram_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=1,
    epochs=EPOCHS,
    seed=SEED,
    workers=1
)

print("Skip-gram training completed!")
print("Vocabulary size:", len(skipgram_model.wv))
print("Vector dimensions:", skipgram_model.wv.vector_size)

Skip-gram training completed!
Vocabulary size: 9527
Vector dimensions: 100


In [39]:
comparison = pd.DataFrame({
    "Parameter": [
        "Training objective",
        "Vector size",
        "Window",
        "Minimum count",
        "Epochs",
        "Vocabulary size"
    ],
    "CBOW": [
        "Context to centre",
        cbow_model.vector_size,
        cbow_model.window,
        cbow_model.min_count,
        cbow_model.epochs,
        len(cbow_model.wv)
    ],
    "Skip-gram": [
        "Centre to context",
        skipgram_model.vector_size,
        skipgram_model.window,
        skipgram_model.min_count,
        skipgram_model.epochs,
        len(skipgram_model.wv)
    ]
})

display(comparison)

,Parameter,CBOW,Skip-gram
0,Training objective,Context to centre,Centre to context
1,Vector size,100,100
2,Window,5,5
3,Minimum count,2,2
4,Epochs,20,20
5,Vocabulary size,9527,9527


In [40]:
query_word = "python"

if (
    query_word in cbow_model.wv.key_to_index
    and query_word in skipgram_model.wv.key_to_index
):
    print("CBOW similar words:")
    print(cbow_model.wv.most_similar(query_word, topn=5))

    print("\nSkip-gram similar words:")
    print(skipgram_model.wv.most_similar(query_word, topn=5))
else:
    print("Query word is missing from one or both vocabularies.")

CBOW similar words:
[('sql', 0.8900838494300842), ('scripting', 0.8440678715705872), ('java', 0.8321564793586731), ('tableau', 0.8281296491622925), ('powerbi', 0.8036924600601196)]

Skip-gram similar words:
[('vba', 0.8095372319221497), ('java', 0.8041990995407104), ('scripting', 0.8026061654090881), ('lawson', 0.7960274815559387), ('tableau', 0.7944450378417969)]


## B3: CBOW vs Skip-gram Training Objectives

### CBOW

Continuous Bag of Words predicts a centre word from its
surrounding context words.

For example, given "python", "developer", "machine",
and "learning", CBOW attempts to predict the missing
centre word.

### Skip-gram

Skip-gram uses a centre word to predict surrounding
context words.

For example, given "developer", Skip-gram attempts to
predict words that frequently occur near it.

### Expected Performance

CBOW generally trains faster and can learn useful
representations for frequent words.

Skip-gram may be better at learning relationships for
less frequent words because it learns from individual
centre-context training pairs.

For our job-description corpus, we hypothesize that
Skip-gram may provide useful representations for
less frequent technical skills.

However, the dataset may be too small to learn strong
semantic relationships, so we will evaluate this
hypothesis using similarity tests in Part C.

In [42]:
fasttext_model = FastText(
    sentences=tokenized_corpus,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=1,
    epochs=EPOCHS,
    seed=SEED,
    workers=1,
    min_n=3,
    max_n=6
)

print("fastText training completed!")
print("Vocabulary size:", len(fasttext_model.wv))
print("Vector dimensions:", fasttext_model.wv.vector_size)

fastText training completed!
Vocabulary size: 9527
Vector dimensions: 100


In [43]:
test_word = "developper"

print("Test word:", test_word)

print(
    "In Word2Vec vocabulary:",
    test_word in skipgram_model.wv.key_to_index
)

print(
    "In fastText vocabulary:",
    test_word in fasttext_model.wv.key_to_index
)

Test word: developper
In Word2Vec vocabulary: False
In fastText vocabulary: False


In [44]:
# Word2Vec OOV behavior
try:
    vector = skipgram_model.wv[test_word]
    print("Word2Vec vector shape:", vector.shape)
except KeyError:
    print("Word2Vec cannot generate a vector for this OOV word.")

# fastText OOV behavior
try:
    vector = fasttext_model.wv[test_word]
    print("fastText vector shape:", vector.shape)
    print("First 10 values:", vector[:10])
except KeyError:
    print("fastText could not generate a vector.")

Word2Vec cannot generate a vector for this OOV word.
fastText vector shape: (100,)
First 10 values: [-0.19777842  0.00475971  0.39883998  0.03281863  0.2625124   0.16247186
 -0.17059126 -0.10814977  0.00462365 -0.46004435]


In [45]:
correct_word = "developer"
misspelled_word = "developper"

if correct_word in fasttext_model.wv.key_to_index:
    similarity = fasttext_model.wv.similarity(
        correct_word,
        misspelled_word
    )

    print("Correct word:", correct_word)
    print("Misspelled word:", misspelled_word)
    print(f"fastText cosine similarity: {similarity:.4f}")
else:
    print("The correct word is not in the fastText vocabulary.")

Correct word: developer
Misspelled word: developper
fastText cosine similarity: 0.7767


In [46]:
from pathlib import Path

# Resolve the project root from the notebook folder
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

cbow_model.save(str(models_dir / "word2vec_cbow.model"))
skipgram_model.save(str(models_dir / "word2vec_skipgram.model"))
fasttext_model.save(str(models_dir / "fasttext.model"))

print("All three models saved to:", models_dir)

All three models saved to: C:\Users\HP\Desktop\Job_Skill_NLP\models


In [47]:
from gensim.models import Word2Vec, FastText

loaded_cbow = Word2Vec.load(
    str(models_dir / "word2vec_cbow.model")
)

loaded_skipgram = Word2Vec.load(
    str(models_dir / "word2vec_skipgram.model")
)

loaded_fasttext = FastText.load(
    str(models_dir / "fasttext.model")
)

print("CBOW loaded:", len(loaded_cbow.wv), "words")
print("Skip-gram loaded:", len(loaded_skipgram.wv), "words")
print("fastText loaded:", len(loaded_fasttext.wv), "words")

CBOW loaded: 9527 words
Skip-gram loaded: 9527 words
fastText loaded: 9527 words


## Part C: Probing and Comparison

### C1: Most Similar Words

We evaluate the semantic relationships learned by CBOW,
Skip-gram and fastText using job-related query words.

In [48]:
query_words = [
    "python",
    "developer",
    "data",
    "analyst",
    "software"
]

print("Selected query words:")
print(query_words)

Selected query words:
['python', 'developer', 'data', 'analyst', 'software']


In [49]:
models = {
    "CBOW": cbow_model,
    "Skip-gram": skipgram_model,
    "fastText": fasttext_model
}

for model_name, model in models.items():
    print(f"\n{model_name}:")

    for word in query_words:
        exists = word in model.wv.key_to_index
        print(f"{word:15s} {exists}")


CBOW:
python          True
developer       True
data            True
analyst         True
software        True

Skip-gram:
python          True
developer       True
data            True
analyst         True
software        True

fastText:
python          True
developer       True
data            True
analyst         True
software        True


In [50]:
for query in query_words:

    print("\n" + "=" * 50)
    print("QUERY WORD:", query)
    print("=" * 50)

    for model_name, model in models.items():

        print(f"\n{model_name}:")

        if query in model.wv.key_to_index:

            similar = model.wv.most_similar(
                query,
                topn=5
            )

            for word, score in similar:
                print(f"{word:20s} {score:.4f}")

        else:
            print("Query word not present in vocabulary.")


QUERY WORD: python

CBOW:
sql                  0.8901
scripting            0.8441
java                 0.8322
tableau              0.8281
powerbi              0.8037

Skip-gram:
vba                  0.8095
java                 0.8042
scripting            0.8026
lawson               0.7960
tableau              0.7944

fastText:
tableau              0.8515
c++                  0.8203
vba                  0.8113
java                 0.8022
jenkin               0.7942

QUERY WORD: developer

CBOW:
architect            0.6461
hana                 0.6204
s/4                  0.5824
gcp                  0.5707
engineering          0.5619

Skip-gram:
configurator         0.5865
practices.collaborate 0.5846
contracting.ability  0.5440
architectsalesforce  0.5434
servicenow           0.5364

fastText:
developed            0.8849
developing           0.7226
envelope             0.6538
epc                  0.6454
dev                  0.6325

QUERY WORD: data

CBOW:
datum                0.7578
aut

### C1: Results and Interpretation

The three trained embedding models were evaluated using
five job-related query words.

For each query, the top five nearest words were retrieved.

The results were examined to determine whether the
neighbouring words were related to the original query.

Words with meaningful relationships indicate that the
model has learned useful contextual patterns.

Some unrelated or unexpected neighbours may occur because
of limited dataset size, word frequency and the context
in which words appear in job descriptions.

The quality of the results is therefore interpreted
cautiously.

In [51]:
word_pairs = [
    ("python", "programming"),
    ("data", "analyst"),
    ("python", "marketing"),
    ("database", "designer")
]

for model_name, model in models.items():

    print("\n" + "=" * 55)
    print(model_name)
    print("=" * 55)

    for word1, word2 in word_pairs:

        if (
            word1 in model.wv.key_to_index
            and word2 in model.wv.key_to_index
        ):

            score = model.wv.similarity(word1, word2)

            print(
                f"{word1:15s} {word2:15s} "
                f"Similarity: {score:.4f}"
            )

        else:
            print(
                f"{word1} - {word2}: "
                "One or both words are unavailable."
            )


CBOW
python          programming     Similarity: 0.7248
data            analyst         Similarity: 0.2112
python          marketing       Similarity: 0.1180
database        designer        Similarity: 0.1671

Skip-gram
python          programming     Similarity: 0.5489
data            analyst         Similarity: 0.3382
python          marketing       Similarity: 0.2942
database        designer        Similarity: 0.1442

fastText
python          programming     Similarity: 0.5812
data            analyst         Similarity: 0.3191
python          marketing       Similarity: 0.3938
database        designer        Similarity: 0.3536


### C2: Cosine Similarity Analysis

Cosine similarity was used to measure the relationship
between selected word pairs.

Two pairs were selected because they were expected to
be related, while two pairs were selected as examples
of less-related concepts.

The similarity scores were compared across CBOW,
Skip-gram and fastText.

The results demonstrate that semantic similarity
depends on the contexts in which words appear in the
training corpus.

Unexpected scores may occur because of limited
training data and differences in word frequency.

In [52]:
# Word analogy using CBOW

try:
    result = cbow_model.wv.most_similar(
        positive=["python", "database"],
        negative=["programming"],
        topn=5
    )

    print("CBOW analogy results:")

    for word, score in result:
        print(f"{word:20s} {score:.4f}")

except KeyError as e:
    print("A required word is missing:", e)

CBOW analogy results:
sql                  0.6594
oracle               0.6560
crm                  0.6415
spreadsheet          0.6268
server               0.6200


In [53]:
try:
    result = skipgram_model.wv.most_similar(
        positive=["python", "database"],
        negative=["programming"],
        topn=5
    )

    print("Skip-gram analogy results:")

    for word, score in result:
        print(f"{word:20s} {score:.4f}")

except KeyError as e:
    print("A required word is missing:", e)

Skip-gram analogy results:
sql                  0.6454
lawson               0.6278
experienceexperience 0.6265
hyperion             0.6244
paychex              0.6171


In [54]:
try:
    result = fasttext_model.wv.most_similar(
        positive=["python", "database"],
        negative=["programming"],
        topn=5
    )

    print("fastText analogy results:")

    for word, score in result:
        print(f"{word:20s} {score:.4f}")

except KeyError as e:
    print("A required word is missing:", e)

fastText analogy results:
sql                  0.6907
dataraptor           0.6853
databrick            0.6584
tableau              0.6494
nosql                0.6420


### C3: Word Analogy Analysis

A word analogy experiment was conducted using the
vector arithmetic operation:

Python - Programming + Database

The experiment was performed using CBOW, Skip-gram
and fastText.

The resulting words were examined to determine whether
the models captured meaningful relationships.

Because the corpus consists of job descriptions rather
than a large general-language dataset, the analogy
results may be weak or unexpected.

An unsuccessful analogy is also reported honestly
because it provides evidence about the limitations
of the learned representations.

In [55]:
comparison_table = pd.DataFrame({
    "Representation": [
        "Bag of Words",
        "TF-IDF",
        "Word2Vec CBOW",
        "Word2Vec Skip-gram",
        "fastText"
    ],

    "Type": [
        "Count-based",
        "Count-based",
        "Learned embedding",
        "Learned embedding",
        "Learned embedding"
    ],

    "Dimensionality": [
        bow_matrix.shape[1],
        tfidf_matrix.shape[1],
        cbow_model.wv.vector_size,
        skipgram_model.wv.vector_size,
        fasttext_model.wv.vector_size
    ],

    "Representation": [
        "Word counts",
        "Word importance",
        "Dense word vectors",
        "Dense word vectors",
        "Dense subword-aware vectors"
    ],

    "OOV Handling": [
        "No learned OOV representation",
        "No learned OOV representation",
        "No",
        "No",
        "Yes, via subword information"
    ]
})

display(comparison_table)

,Representation,Type,Dimensionality,OOV Handling
0,Word counts,Count-based,18635,No learned OOV representation
1,Word importance,Count-based,18635,No learned OOV representation
2,Dense word vectors,Learned embedding,100,No
3,Dense word vectors,Learned embedding,100,No
4,Dense subword-aware vectors,Learned embedding,100,"Yes, via subword information"


In [56]:
print("BoW sparsity:", f"{sparsity:.2f}%")
print("TF-IDF sparsity:", f"{tfidf_sparsity:.2f}%")

print("\nEmbedding vector dimensions:")
print("CBOW:", cbow_model.wv.vector_size)
print("Skip-gram:", skipgram_model.wv.vector_size)
print("fastText:", fasttext_model.wv.vector_size)

BoW sparsity: 98.60%
TF-IDF sparsity: 98.60%

Embedding vector dimensions:
CBOW: 100
Skip-gram: 100
fastText: 100


### C4: Count-Based vs Learned Representations

Count-based representations such as Bag of Words and
TF-IDF represent documents using vocabulary-based
features.

Their dimensionality depends on the number of unique
features in the corpus. These matrices are generally
sparse because most words do not occur in every
document.

Word2Vec and fastText learn dense word representations
using contextual information.

CBOW and Skip-gram use surrounding words to learn
word vectors, while fastText additionally uses
character-level n-grams.

Unlike standard Word2Vec, fastText can generate
representations for many words that were not observed
as complete vocabulary entries.

Count-based methods provide explicit word-frequency
and importance information, while learned embeddings
capture contextual relationships between words.

The effectiveness of each representation depends
on the size and diversity of the job-description corpus.

# Part D: Packaging Word Representations

This section saves the trained TF-IDF vectorizer and
embedding models for reuse in Phase 3.

A reusable function is implemented to convert a raw
job description into a numerical vector using a
selected representation.

In [57]:
import joblib
from pathlib import Path

# Find the project root
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Create models directory if necessary
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Save the already-fitted TF-IDF vectorizer
joblib.dump(
    tfidf_vectorizer,
    models_dir / "tfidf_vectorizer.pkl"
)

print("TF-IDF vectorizer saved successfully!")
print("Saved at:", models_dir / "tfidf_vectorizer.pkl")

TF-IDF vectorizer saved successfully!
Saved at: C:\Users\HP\Desktop\Job_Skill_NLP\models\tfidf_vectorizer.pkl


In [58]:
print((models_dir / "tfidf_vectorizer.pkl").exists())

True


In [59]:
model_files = [
    "tfidf_vectorizer.pkl",
    "word2vec_cbow.model",
    "word2vec_skipgram.model",
    "fasttext.model"
]

for filename in model_files:
    file_path = models_dir / filename

    if file_path.exists():
        print(f"FOUND: {filename}")
    else:
        print(f"MISSING: {filename}")

FOUND: tfidf_vectorizer.pkl
FOUND: word2vec_cbow.model
FOUND: word2vec_skipgram.model
FOUND: fasttext.model


In [61]:
import sys

# Make the src folder importable
src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from vectorizer import DocumentVectorizer

# Load all saved representations
document_vectorizer = DocumentVectorizer()

print("All models loaded successfully!")

All models loaded successfully!


In [62]:
sample_text = """
We are hiring a Python developer with experience
in machine learning, data analysis, and SQL databases.
"""

representations = [
    "tfidf",
    "cbow",
    "skipgram",
    "fasttext"
]

for representation in representations:

    vector = document_vectorizer.vectorize_document(
        sample_text,
        representation
    )

    print("\nRepresentation:", representation)
    print("Vector shape:", vector.shape)

    if representation == "tfidf":
        print("Nonzero features:", vector.nnz)
    else:
        print("First 10 values:", vector[:10])


Representation: tfidf
Vector shape: (1, 18635)
Nonzero features: 15

Representation: cbow
Vector shape: (100,)
First 10 values: [ 0.31384975  0.07417448 -0.4447993   0.34617642 -0.08573157 -0.07782713
 -0.47054657 -0.9619072   0.0142743   0.3168758 ]

Representation: skipgram
Vector shape: (100,)
First 10 values: [-0.04979673  0.11277908 -0.28555632  0.3171039  -0.01298479  0.27110913
 -0.20165297  0.11886737 -0.07666195 -0.05099085]

Representation: fasttext
Vector shape: (100,)
First 10 values: [ 0.03894644  0.07402716  0.18483552  0.08422992  0.28800112  0.09389632
 -0.04617058 -0.1395551   0.16113074 -0.21115468]


In [63]:
empty_text = ""

for representation in representations:
    vector = document_vectorizer.vectorize_document(
        empty_text,
        representation
    )

    print(representation, ":", vector.shape)

tfidf : (1, 18635)
cbow : (100,)
skipgram : (100,)
fasttext : (100,)
